### Harry potter rag 

In [1]:
from dotenv import load_dotenv
from pathlib import Path

import os
import pandas as pd

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

csv_files = [
    "cleaned_characters.csv",
    "cleaned_spells.csv",
    "cleaned_potions.csv",
]

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=api_key
)

/Users/opheliathomasson/Documents/TUC_datamanager/datakvalité/kunskapskontroll/data_kvalit-_kunskapskontroll/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Example question to test the models response and to compare it to the retrieved context answer.

question = "What side effects does Felix Felicis have?"

response = model.invoke(question)

print(response.content)

In [ ]:
# Create documents from the CSV files and 
from langchain_core.documents import Document

documents = []

for file_name in csv_files:

    file_path = Path("data") / file_name

    df = pd.read_csv(file_path)

    for _, row in df.iterrows():

        fields = []

        for col in df.columns:

            value = row[col]

            if pd.notna(value) and str(value).strip() not in ["", "[]", "nan"]:

                fields.append(f"{col}: {value}")

        content = "\n".join(fields)

        documents.append(
            Document(
                page_content=content,
                metadata={
                    "source": file_name,
                    "type": "csv_row",
                    "name": row.get("name", "")
                }
            )
        )

print(f"Number of documents: {len(documents)}")
print(documents[0].page_content)

Number of documents: 513
id: 1
name: Harry James Potter
gender: Male
job: Student
house: Gryffindor
patronus: Stag
species: Human
blood_status: Half-blood
loyalty: Albus Dumbledore | Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
skills: Parseltongue| Defence Against the Dark Arts | Seeker
birth: 31 July 1980


In [5]:
# We will also load the text files in the "data" directory and convert them into Document objects as well, so that we can use them for retrieval and question answering.
from pathlib import Path
from langchain_community.document_loaders import TextLoader

book_documents = []

for file_path in Path("data").glob("*.txt"): # Loop through all text files in the "data" directory
    
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()

    for doc in docs:
        doc.metadata["source"] = file_path.name

    book_documents.extend(docs)

print(len(book_documents))

7


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

book_chunks = text_splitter.split_documents(book_documents)

print(len(book_chunks))

10965


In [7]:
all_documents = documents + book_chunks # Combine the documents from the CSV file and the text files into a single list

In [8]:
# use the HuggingFaceEmbeddings class from langchain_community to create an embeddings model instead of the GoogleGenerativeAIEmbeddings 
# class from langchain_google_genai, since the latter is not working because the limit for the free API key has been reached. The HuggingFaceEmbeddings class 
# allows us to use a variety of pre-trained models from Hugging Face, including the sentence-transformers models, which are designed for generating embeddings for 
# sentences and paragraphs.

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/p5/jkbyc74n1jj4grxpk0w1ltf80000gn/T/ipykernel_7739/1062252735.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6588.68it/s]


In [9]:
# Create a Chroma vector store and add the documents to it
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="harry_potter_hf",
    embedding_function=embeddings,
    persist_directory="./chroma_harry_potter_many_db",
)

In [10]:
# Entering documents into the vector data base
batch_size = 500

for i in range(0, len(all_documents), batch_size):
    batch = all_documents[i:i + batch_size]

    vector_store.add_documents(documents=batch)

    print(f"Added documents {i} to {i + len(batch)}")

Added documents 0 to 500
Added documents 500 to 1000
Added documents 1000 to 1500
Added documents 1500 to 2000
Added documents 2000 to 2500
Added documents 2500 to 3000
Added documents 3000 to 3500
Added documents 3500 to 4000
Added documents 4000 to 4500
Added documents 4500 to 5000
Added documents 5000 to 5500
Added documents 5500 to 6000
Added documents 6000 to 6500
Added documents 6500 to 7000
Added documents 7000 to 7500
Added documents 7500 to 8000
Added documents 8000 to 8500
Added documents 8500 to 9000
Added documents 9000 to 9500
Added documents 9500 to 10000
Added documents 10000 to 10500
Added documents 10500 to 11000
Added documents 11000 to 11478


In [11]:
# Checking the similarity search with a sample question.
question = "What house does Harry Potter belong to?"

results = vector_store.similarity_search(question, k=3)

for result in results:
    print(result.page_content)
    print("---")

Page | 75 Harry Potter and the Half Blood Prince - J.K. Rowling 




Muggle house — the owners of this place are on 
holiday in the Canary Islands — it’s been very 
pleasant, I’ll be sorry to leave. It’s quite easy once you 
know how, one simple Freezing Charm on these 
absurd burglar alarms they use instead of 
Sneakoscopes and make sure the neighbors don’t 
spot you bringing in the piano.” 

“Ingenious,” said Dumbledore. “But it sounds a rather 
tiring existence for a broken-down old buffer in 
search of a quiet life. Now, if you were to return to 
Hogwarts — ”
---
Harry understood completely. He knew how he would 
feel if forced, when he was grown up and thought he 
was free of the place forever, to return and live at 
number four, Privet Drive. 

“It’s ideal for headquarters, of course,” Sirius said. 

“My father put every security measure known to 
Wizard-kind on it when he lived here. It’s Unplottable, 
so Muggles could never come and call — as if they’d 
have wanted to — and now

In [13]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5,"fetch_k": 20}) # Using MMR to get more diverse results and using fetch_k to get more results to choose from before re-ranking them with MMR.

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
You are an old and wise wizard from the magical world.
Answer the user's question in a mysterious but helpful tone.

Use only the information in the retrieved data.
If the answer is not found in the retrieved data, say that even your ancient magic cannot find the answer.

Retrieved data:
{retrieved_data}

Question:
{question}
"""


In [15]:
prompt = PromptTemplate.from_template(template)
parser = StrOutputParser()

In [16]:
chain = (
    {
        "retrieved_data": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)

In [17]:
question = input("Ask a question about the Harry Potter dataset: ")


In [18]:
answer = chain.invoke(question)

In [19]:
print(answer)

Ah, a question of time and the turning of years... The scrolls speak of a boy born in the month of July. I see a flickering memory from his eleventh year; his birthday fell upon a Tuesday, following a somber Monday spent trapped in a car by the sea. 

While the month is clear, the specific day of that month remains hidden in the mists, for even my ancient magic cannot find the exact date within these particular records.
